In [1]:
import sqlite3
import requests
import time
import json
import pandas as pd
import urllib.parse
from dotenv import load_dotenv
import os
import datetime
from concurrent.futures import ThreadPoolExecutor

from pathlib import Path

In [2]:
ROOT = Path.cwd().parent


In [3]:
load_dotenv(ROOT / '.env')
ITAD_KEY = os.getenv('ITAD_API_KEY')
DB_PATH = str(ROOT / 'db' / 'gaming_warehouse.db')
ITAD_BASE = 'https://api.isthereanydeal.com'
TIMEOUT   = 15

In [4]:
def crear_tablas():
    with sqlite3.connect(DB_PATH) as conn:

        # Agregar itad_id_texto a CAT_Juego si no existe
        cols = [r[1] for r in conn.execute("PRAGMA table_info(CAT_Juego)").fetchall()]
        if 'itad_id_texto' not in cols:
            conn.execute("ALTER TABLE CAT_Juego ADD COLUMN itad_id_texto TEXT")

        conn.executescript("""
            CREATE TABLE IF NOT EXISTS Hist_Precios_ITAD (
                itad_id_texto TEXT,
                precio_base   REAL,
                precio        REAL,
                descuento     INTEGER,
                fecha_unix    INTEGER,
                PRIMARY KEY (itad_id_texto, fecha_unix)
            );

            CREATE TABLE IF NOT EXISTS Datos_Actuales_ITAD (
                itad_id_texto       TEXT PRIMARY KEY,
                precio_actual       REAL,
                precio_minimo       REAL,
                fecha_minimo        INTEGER,
                en_bundle           INTEGER DEFAULT 0,
                descuento_actual    INTEGER,
                expiry              INTEGER,
                fecha_actualizacion INTEGER
            );
        """)

        # Agregar columnas nuevas a Datos_Actuales_ITAD si no existen
        cols = [r[1] for r in conn.execute("PRAGMA table_info(Datos_Actuales_ITAD)").fetchall()]
        for col, tipo in [("descuento_actual", "INTEGER"), ("expiry", "INTEGER")]:
            if col not in cols:
                conn.execute(f"ALTER TABLE Datos_Actuales_ITAD ADD COLUMN {col} {tipo}")



In [5]:
crear_tablas()


In [ ]:
# Prueba con 10 juegos — ver mapeo antes de insertar
with sqlite3.connect(DB_PATH) as conn:
    muestra = conn.execute("""
        SELECT juego_id, titulo, id_steam
        FROM CAT_Juego
        WHERE id_steam IS NOT NULL
          AND itad_id_texto IS NULL
        LIMIT 10
    """).fetchall()

resultados = []
for juego_id, titulo, id_steam in muestra:
    try:
        res = requests.get(
            f"{ITAD_BASE}/games/lookup/v1",
            params={"key": ITAD_KEY, "appid": id_steam},
            timeout=TIMEOUT
        )
        data = res.json()
        resultados.append({
            "juego_id":juego_id,
            "titulo_db":titulo,
            "id_steam":id_steam,
            "found":data.get("found", False),
            "itad_id":data["game"]["id"]    if data.get("found") else None,
            "titulo_itad": data["game"]["title"] if data.get("found") else None,
        })
    except Exception as e:
        resultados.append({"juego_id": juego_id, "titulo_db": titulo, "id_steam": id_steam, "found": False, "itad_id": None, "titulo_itad": str(e)})
    time.sleep(0.5)

df_mapeo = pd.DataFrame(resultados)
print(f"{df_mapeo['found'].sum()} / {len(df_mapeo)}")
display(df_mapeo)

Encontrados: 10 / 10


,juego_id,titulo_db,id_steam,found,itad_id,titulo_itad
0,1,Grand Theft Auto V,271590,True,018d937f-03e2-7281-a961-037a2d279a92,Grand Theft Auto V
1,2,The Witcher 3: Wild Hunt,292030,True,018d937f-1212-7232-b23f-a046f6fd4a57,The Witcher 3: Wild Hunt
2,3,Portal 2,620,True,018d937f-21e1-728e-86d7-9acb3c59f2bb,Portal 2
3,4,The Elder Scrolls V: Skyrim,72850,True,018d937f-2b23-73a3-9b40-d93860065d00,The Elder Scrolls V: Skyrim
4,5,Grand Theft Auto: San Andreas,12120,True,018d937e-f829-73ff-aa7f-95421948da6b,Grand Theft Auto - San Andreas
5,6,Portal,400,True,018d937f-1749-73f8-bf5a-8e0aa3cdd9ff,Portal
6,7,Red Dead Redemption 2,1174180,True,018d937f-3a3b-7210-bd2d-0d1dfb1d84c0,Red Dead Redemption 2
7,8,God of War,1593500,True,018d937f-4964-7357-a557-f04dac608915,God of War
8,9,Half-Life 2,220,True,018d937f-012f-73b8-ab2c-898516969e6a,Half-Life 2
9,10,BioShock,7670,True,018d937f-00db-709e-9c93-c0c25174ca30,BioShock


In [9]:
def vincular_juegos(limite=100):
    with sqlite3.connect(DB_PATH) as conn:
        pendientes = conn.execute("""
            SELECT juego_id, titulo, id_steam
            FROM CAT_Juego
            WHERE id_steam IS NOT NULL
              AND itad_id_texto IS NULL
            LIMIT ?
        """, (limite,)).fetchall()

    if not pendientes:
        print("Sin pendientes, al fin acabo")
        return 0

    total     = len(pendientes)
    vinculados = 0
    print(f"{total} juegos a vincular")

    for i, (juego_id, titulo, id_steam) in enumerate(pendientes, 1):
        try:
            res = requests.get(
                f"{ITAD_BASE}/games/lookup/v1",
                params={"key": ITAD_KEY, "appid": id_steam},
                timeout=TIMEOUT
            )
            if res.status_code == 429:
                print("limt, esperando 20s")
                time.sleep(20)
                continue

            data = res.json()
            if data.get("found"):
                itad_id = data["game"]["id"]
                with sqlite3.connect(DB_PATH) as conn:
                    conn.execute(
                        "UPDATE CAT_Juego SET itad_id_texto = ? WHERE juego_id = ?",
                        (itad_id, juego_id)
                    )
                vinculados += 1
                print(f"  [{i}/{total}] OK: {titulo} → {itad_id}")
            else:
                print(f"  [{i}/{total}] no encontrado: {titulo}")

        except KeyboardInterrupt:
            print(f"\nInterrumpido en [{i}/{total}]. Vinculados hasta ahora: {vinculados}")
            return vinculados
        except Exception as e:
            print(f"  [{i}/{total}] error [{titulo}]: {e}")
        finally:
            time.sleep(0.5)

    return vinculados

In [10]:
while vincular_juegos(limite=100) > 0:
    time.sleep(2)

100 juegos a vincular
  [1/100] OK: Grand Theft Auto V → 018d937f-03e2-7281-a961-037a2d279a92
  [2/100] OK: The Witcher 3: Wild Hunt → 018d937f-1212-7232-b23f-a046f6fd4a57
  [3/100] OK: Portal 2 → 018d937f-21e1-728e-86d7-9acb3c59f2bb
  [4/100] OK: The Elder Scrolls V: Skyrim → 018d937f-2b23-73a3-9b40-d93860065d00
  [5/100] OK: Grand Theft Auto: San Andreas → 018d937e-f829-73ff-aa7f-95421948da6b
  [6/100] OK: Portal → 018d937f-1749-73f8-bf5a-8e0aa3cdd9ff
  [7/100] OK: Red Dead Redemption 2 → 018d937f-3a3b-7210-bd2d-0d1dfb1d84c0
  [8/100] OK: God of War → 018d937f-4964-7357-a557-f04dac608915
  [9/100] OK: Half-Life 2 → 018d937f-012f-73b8-ab2c-898516969e6a
  [10/100] OK: BioShock → 018d937f-00db-709e-9c93-c0c25174ca30
  [11/100] OK: Assassin's Creed II → 018d937f-3497-71d8-91b5-442e6b36f0cd
  [12/100] OK: Grand Theft Auto: Vice City → 018d937e-f01b-70c9-85e2-52a3122c9560
  [13/100] OK: Half-Life → 018d937f-0bc2-7120-8fb2-a469fde4fcbf
  [14/100] OK: Mass Effect 2 → 018d937f-6d20-7059-905f-

In [6]:
with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql("""
        SELECT 
            COUNT(*)                                    AS total,
            COUNT(itad_id_texto)                        AS vinculados,
            COUNT(*) - COUNT(itad_id_texto)             AS sin_vincular
        FROM CAT_Juego
        WHERE id_steam IS NOT NULL
    """, conn)
    display(df)

,total,vinculados,sin_vincular
0,7181,7172,9


In [7]:
with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql("""
        SELECT juego_id, titulo, id_steam,categoria
        FROM CAT_Juego
        WHERE id_steam IS NOT NULL
          AND itad_id_texto IS NULL
    """, conn)
    display(df)

,juego_id,titulo,id_steam,categoria
0,73,Counter-Strike: Global Offensive,4465480,Juego principal
1,1274,Street Fighter 6,1952120,Juego principal
2,1419,FIFA 23,1962391,Juego principal
3,1614,Warface,3889960,Juego principal
4,2750,Tomb Raider: Game of the Year Edition,379180,Bundle
5,3532,SpaceChem,775011,Juego principal
6,4642,Men of War: Assault Squad 2,562040,Juego principal
7,8079,Team Fortress 2 Classified,3545060,Mod
8,9182,0 A.D.,2158440,Juego principal


# obtener precios

In [8]:
res = requests.get(
    f"{ITAD_BASE}/service/shops/v1",
    params={"country": "US"},
    timeout=TIMEOUT
)
print(res.status_code)
shops = res.json()
df_shops = pd.DataFrame(shops)
display(df_shops[df_shops['title'].str.contains('Steam', case=False, na=False)])

200


,id,title,deals,games,update
28,61,Steam,23958,268319,2026-04-10T23:00:59+02:00


In [9]:
STEAM_SHOP_ID = 61

In [10]:
def fetch_historial_steam(itad_id, meses_atras=24):
    since = datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(days=meses_atras * 30)
    
    res = requests.get(
        f"{ITAD_BASE}/games/history/v2",
        params={
            "key":   ITAD_KEY,
            "id":    itad_id,
            "shops": STEAM_SHOP_ID,
            "since": since.strftime('%Y-%m-%dT%H:%M:%SZ')
        },
        timeout=TIMEOUT
    )
    if res.status_code != 200:
        return []

    return [
        {
            "precio_base": c["deal"]["regular"]["amount"],
            "precio":      c["deal"]["price"]["amount"],
            "descuento":   c["deal"]["cut"],
            "fecha_unix":  int(datetime.datetime.fromisoformat(c["timestamp"]).timestamp())
        }
        for c in res.json()
    ]

In [11]:
def buscar_itad(titulo):
    res = requests.get(
        f"{ITAD_BASE}/games/search/v1",
        params={"key": ITAD_KEY, "title": titulo, "limit": 5},
        timeout=TIMEOUT
    )
    for r in res.json():
        print(f" {r['id']} — {r['title']}")

In [12]:
buscar_itad("resident evil 4")


 018d937f-6b99-7077-9cf6-eacfce97587f — Resident Evil 4 (2005)
 018d937f-74c1-708e-8193-97dfbe6a98fe — Resident Evil 4 (2023)
 018d937f-751c-72ae-af51-cc2ae4a8726f — Resident Evil 4 - Separate Ways
 018d937f-32cf-736d-a8fb-0e2a4b3b3d4c — Resident Evil 0 Costume Pack 4
 018d937f-6c9f-720e-85fb-da2c0428a2bb — Resident Evil 4 - The Mercenaries
 018f084f-a7d1-7026-ba37-f11927b55307 — Resident Evil 4 (2023) - Gold Edition
 018d937f-6bee-71a1-89df-c91a7026c2b4 — Resident Evil 4 Deluxe Weapon: 'Skull Shaker'
 018d937f-6bee-71a1-89df-c91a6f3c9598 — Resident Evil 4 Deluxe Weapon: 'Sentinel Nine'
 018d937f-6bee-71a1-89df-c91a6e486b15 — Resident Evil 4 Leon Accessory: 'Sunglasses (Sporty)'
 018d937f-6bee-71a1-89df-c91a6d3cc51a — Resident Evil 4 'Original Ver.' Soundtrack Swap
 018d937f-6bee-71a1-89df-c91a6d3b63ac — Resident Evil 4 Leon Costume & Filter: 'Hero'
 018d937f-6bed-737d-b6d8-9bf0b165ad50 — Resident Evil 4 Leon & Ashley Costumes: 'Casual'
 018d937f-6bed-737d-b6d8-9bf0b118196d — Resident 

In [13]:
historial = fetch_historial_steam("018d937f-74c1-708e-8193-97dfbe6a98fe", meses_atras=60)
print(f"{len(historial)} registros")
for r in historial:
    print(r)

51 registros
{'precio_base': 39.99, 'precio': 15.99, 'descuento': 60, 'fecha_unix': 1775063776}
{'precio_base': 39.99, 'precio': 39.99, 'descuento': 0, 'fecha_unix': 1772129990}
{'precio_base': 39.99, 'precio': 15.99, 'descuento': 60, 'fecha_unix': 1770920579}
{'precio_base': 39.99, 'precio': 39.99, 'descuento': 0, 'fecha_unix': 1767641357}
{'precio_base': 39.99, 'precio': 15.99, 'descuento': 60, 'fecha_unix': 1765477169}
{'precio_base': 39.99, 'precio': 39.99, 'descuento': 0, 'fecha_unix': 1762881334}
{'precio_base': 39.99, 'precio': 19.99, 'descuento': 50, 'fecha_unix': 1761758251}
{'precio_base': 39.99, 'precio': 39.99, 'descuento': 0, 'fecha_unix': 1759774647}
{'precio_base': 39.99, 'precio': 19.99, 'descuento': 50, 'fecha_unix': 1758647934}
{'precio_base': 39.99, 'precio': 39.99, 'descuento': 0, 'fecha_unix': 1756055979}
{'precio_base': 39.99, 'precio': 19.99, 'descuento': 50, 'fecha_unix': 1754932916}
{'precio_base': 39.99, 'precio': 39.99, 'descuento': 0, 'fecha_unix': 175217089

In [14]:
res = requests.post(
    f"{ITAD_BASE}/games/prices/v3",
    params={"key": ITAD_KEY, "country": "US", "shops": STEAM_SHOP_ID},
    json=["018d937f-74c1-708e-8193-97dfbe6a98fe"],
    timeout=TIMEOUT
)
print(res.status_code)
print(json.dumps(res.json(), indent=2))

200
[
  {
    "id": "018d937f-74c1-708e-8193-97dfbe6a98fe",
    "historyLow": {
      "all": {
        "amount": 9.6,
        "amountInt": 960,
        "currency": "USD"
      },
      "y1": {
        "amount": 12.9,
        "amountInt": 1290,
        "currency": "USD"
      },
      "m3": {
        "amount": 13.59,
        "amountInt": 1359,
        "currency": "USD"
      }
    },
    "deals": [
      {
        "shop": {
          "id": 61,
          "name": "Steam"
        },
        "price": {
          "amount": 15.99,
          "amountInt": 1599,
          "currency": "USD"
        },
        "regular": {
          "amount": 39.99,
          "amountInt": 3999,
          "currency": "USD"
        },
        "cut": 60,
        "voucher": null,
        "storeLow": {
          "amount": 15.99,
          "amountInt": 1599,
          "currency": "USD"
        },
        "flag": "S",
        "drm": [],
        "platforms": [
          {
            "id": 1,
            "name": "Windows"

In [15]:
def fetch_precio_actual(itad_id):
    res = requests.post(
        f"{ITAD_BASE}/games/prices/v3",
        params={"key": ITAD_KEY, "country": "US", "shops": STEAM_SHOP_ID},
        json=[itad_id],
        timeout=TIMEOUT
    )
    if res.status_code != 200 or not res.json():
        return None, None, None, None, None

    data  = res.json()[0]
    deals = data.get("deals", [])

    precio_actual    = deals[0]["price"]["amount"]                                          if deals else None
    descuento_actual = deals[0]["cut"]                                                      if deals else None
    expiry           = int(datetime.datetime.fromisoformat(deals[0]["expiry"]).timestamp()) if deals and deals[0].get("expiry") else None
    precio_minimo    = deals[0]["storeLow"]["amount"]                                       if deals and deals[0].get("storeLow") else None
    fecha_minimo     = int(datetime.datetime.fromisoformat(deals[0]["timestamp"]).timestamp()) if deals else None

    return precio_actual, descuento_actual, expiry, precio_minimo, fecha_minimo

In [16]:
print(fetch_precio_actual("018d937f-74c1-708e-8193-97dfbe6a98fe"))

(15.99, 60, 1776272400, 15.99, 1775063776)


In [17]:
def procesar_lote(meses_atras=12, limite=100):
    with sqlite3.connect(DB_PATH) as conn:
        pendientes = conn.execute("""
            SELECT j.itad_id_texto
            FROM CAT_Juego j
            LEFT JOIN Datos_Actuales_ITAD d ON j.itad_id_texto = d.itad_id_texto
            WHERE j.itad_id_texto IS NOT NULL
              AND d.itad_id_texto IS NULL
            LIMIT ?
        """, (limite,)).fetchall()

    if not pendientes:
        print("Sin pendientes.")
        return 0

    total = len(pendientes)
    print(f"{total} juegos a procesar")

    for i, (itad_id,) in enumerate(pendientes, 1):
        try:
            historial                                              = fetch_historial_steam(itad_id, meses_atras)
            precio_actual, descuento_actual, expiry, precio_minimo, fecha_minimo = fetch_precio_actual(itad_id)

            with sqlite3.connect(DB_PATH) as conn:
                if historial:
                    conn.executemany("""
                        INSERT OR IGNORE INTO Hist_Precios_ITAD
                        (itad_id_texto, precio_base, precio, descuento, fecha_unix)
                        VALUES (?, ?, ?, ?, ?)
                    """, [(itad_id, r["precio_base"], r["precio"], r["descuento"], r["fecha_unix"]) for r in historial])

                conn.execute("""
                    INSERT OR REPLACE INTO Datos_Actuales_ITAD
                    (itad_id_texto, precio_actual, precio_minimo, fecha_minimo,
                     descuento_actual, expiry, fecha_actualizacion)
                    VALUES (?, ?, ?, ?, ?, ?, strftime('%s','now'))
                """, (itad_id, precio_actual, precio_minimo, fecha_minimo, descuento_actual, expiry))

            print(f"  [{i}/{total}] {itad_id} — hist: {len(historial)} | actual: {precio_actual} | min: {precio_minimo}")

        except KeyboardInterrupt:
            print(f"\nInterrumpido en [{i}/{total}].")
            return i
        except Exception as e:
            print(f"  [{i}/{total}] error {itad_id}: {e}")

        time.sleep(0.5)

    return total

In [ ]:
# with sqlite3.connect(DB_PATH) as conn:
#     conn.executescript("""
#         DELETE FROM Hist_Precios_ITAD;
#         DELETE FROM Datos_Actuales_ITAD;
#     """)
# print("Tablas limpias.")

Tablas limpias.


In [ ]:
while procesar_lote(meses_atras=12, limite=100) > 0:
    time.sleep(5)
print("Todos procesados.")

100 juegos a procesar
  [1/100] 018d937f-07c2-7077-bebc-928c09f137d8 — hist: 18 | actual: 29.99 | min: 4.49
  [2/100] 018d937f-222d-7300-b046-874f2a8d710f — hist: 0 | actual: None | min: None
  [3/100] 018d937f-105d-72c2-ac96-b538ae0967f1 — hist: 13 | actual: 19.99 | min: 1.99
  [4/100] 018d937f-21da-71a5-b774-323d7a867ff8 — hist: 18 | actual: 5.99 | min: 2.99
  [5/100] 018d937f-3cc5-7116-b8e1-06ca7dd2e7ca — hist: 17 | actual: 29.99 | min: 5.99
  [6/100] 018d937e-fd5e-7114-937e-5357a75217fc — hist: 19 | actual: 14.99 | min: 1.49
  [7/100] 018d937f-4640-728f-9dee-ff5702d93303 — hist: 0 | actual: None | min: None
  [8/100] 018d937f-2067-73c1-912d-8192a3f11244 — hist: 18 | actual: 9.99 | min: 2.49
  [9/100] 018d937e-f9c8-727a-ad7b-cb2c440e3e66 — hist: 23 | actual: 9.99 | min: 0.99
  [10/100] 018d9589-dd61-71cb-b314-c914cb0d2b0e — hist: 19 | actual: 9.99 | min: 1.99
  [11/100] 018d937f-0165-7011-8fe2-f82705e769d8 — hist: 6 | actual: 0 | min: 3.99
  [12/100] 018d937f-1b60-7380-8a75-fd184ab2

In [21]:
with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql("""
        SELECT *
        FROM CAT_Juego
        WHERE titulo like "%Tomb Raider%"
    """, conn)
    display(df)

,juego_id,id_igdb,id_steam,titulo,categoria,fecha_lanzamiento,resumen,historia,url_portada,puntuacion_igdb,...,hltb_completacionista,steam_price_initial,steam_price_final,steam_discount_percent,metacritic_score,recommendations_count,achievements_count,steam_languages,pc_requirements_json,itad_id_texto
0,24,1164,203160.0,Tomb Raider,Juego principal,2013-03-04,Tomb Raider explores the intense and gritty or...,he game begins with Lara setting out on her fi...,https://images.igdb.com/igdb/image/upload/t_co...,81.760627,...,None,329.00,329.00,0.0,86.0,168219.0,50.0,"Inglés<strong>*</strong>, Alemán<strong>*</str...","{""minimum"": ""<strong>Minimum:</strong><br><ul ...",018d937e-f9f4-7235-b38c-bfdcb325b577
1,74,7323,391220.0,Rise of the Tomb Raider,Juego principal,2015-11-10,Join Lara Croft on her first great tomb raidin...,Sin datos,https://images.igdb.com/igdb/image/upload/t_co...,81.286752,...,None,499.00,499.00,0.0,86.0,122662.0,143.0,"Inglés<strong>*</strong>, Francés<strong>*</st...","{""minimum"": ""<strong>M\u00ednimo:</strong><br>...",018d937e-f4d8-7368-a624-4a0e38d5d2eb
2,198,37777,750920.0,Shadow of the Tomb Raider,Juego principal,2018-09-14,As Lara Croft races to save the world from a M...,In the two months since Rise of the Tomb Raide...,https://images.igdb.com/igdb/image/upload/t_co...,77.546357,...,None,669.00,669.00,0.0,77.0,69797.0,99.0,"Inglés<strong>*</strong>, Francés<strong>*</st...","{""minimum"": ""<strong>M\u00ednimo:</strong><br>...",018d937f-3a11-70d1-81cd-cac58ba1e391
3,305,912,224960.0,Tomb Raider,Juego principal,1996-10-24,Tomb Raider is a 3D action game developed by C...,"Lara Croft is a Tomb Raider, an archaeologist ...",https://images.igdb.com/igdb/image/upload/t_co...,80.777766,...,None,119.00,119.00,0.0,NaN,4005.0,0.0,Inglés<strong>*</strong><br><strong>*</strong>...,"{""minimum"": ""<strong>Minimum:</strong><br><ul ...",018d9573-86c4-70ce-b106-a885d3eb3759
4,443,1156,225300.0,Tomb Raider II,Juego principal,1997-11-21,Tomb Raider II is the second game of the Tomb ...,The story begins one year after the events of ...,https://images.igdb.com/igdb/image/upload/t_co...,80.204248,...,None,119.00,119.00,0.0,NaN,2019.0,0.0,Inglés<strong>*</strong><br><strong>*</strong>...,"{""minimum"": ""<strong>Minimum:</strong><br><ul ...",018d9573-86f5-70ae-b7a2-c82924d055c3
5,504,1161,7000.0,Tomb Raider: Legend,Juego principal,2006-04-07,Tomb Raider: Legend is the seventh major game ...,"Lara is searching for a South American relic, ...",https://images.igdb.com/igdb/image/upload/t_co...,74.816495,...,None,119.00,119.00,0.0,82.0,5203.0,0.0,"Inglés, Francés, Italiano, Alemán, Español de ...","{""minimum"": ""<strong>M\u00ednimo: </strong>Mic...",018d937f-1d16-7246-baee-aa4e147bab19
6,578,1163,8140.0,Tomb Raider: Underworld,Juego principal,2008-11-18,Tomb Raider: Underworld represents a new advan...,Continuing the events from Tomb Raider: Legend...,https://images.igdb.com/igdb/image/upload/t_co...,74.616209,...,None,149.00,149.00,0.0,80.0,5965.0,0.0,"Inglés, Francés, Italiano, Alemán, Español de ...","{""minimum"": ""M\u00ednimo<br><br> ...",018d937f-0e94-7003-b94b-805c96665bfa
7,607,1162,8000.0,Tomb Raider: Anniversary,Remake,2007-06-01,Tomb Raider: Anniversary is a remake of the or...,Lara Croft has been hired by a mysterious woma...,https://images.igdb.com/igdb/image/upload/t_co...,76.473843,...,None,149.00,149.00,0.0,83.0,6440.0,0.0,"Inglés, Francés, Alemán, Italiano, Español de ...","{""minimum"": ""<strong>M\u00ednimo: </strong>Mic...",018d937f-1974-701c-8cb8-e3f33320a0e5
8,806,1157,225320.0,Tomb Raider III: Adventures of Lara Croft,Juego principal,1998-11-20,Tomb Raider III: Adventures of Lara Croft is a...,The intrepid archaeologist Lara Croft ventures...,https://images.igdb.com/igdb/image/upload/t_co...,77.080172,...,None,119.00,119.00,0.0,NaN,1304.0,0.0,Inglés<strong>*</strong><br><strong>*</strong>...,"{""minimum"": ""<strong>Minimum:</strong><br><ul ...",018d9573-8704-7398-b71d-3770d86747d4
9,942,19965,NaN,T

In [13]:
def preparar_tablas_itad():
    conn = sqlite3.connect(DB_PATH)
    try:
        cursor = conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS CAT_ITAD_Juego (
                itad_id_texto TEXT PRIMARY KEY,
                itad_titulo TEXT
            )
        """)
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS REL_Juego_ITAD (
                juego_id INTEGER,
                itad_id_texto TEXT,
                PRIMARY KEY (juego_id, itad_id_texto),
                FOREIGN KEY (juego_id) REFERENCES CAT_Juego(juego_id),
                FOREIGN KEY (itad_id_texto) REFERENCES CAT_ITAD_Juego(itad_id_texto)
            )
        """)
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS CAT_Tienda (
                id_tienda INTEGER PRIMARY KEY,
                nombre TEXT
            )
        """)
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS Hist_Precios_ITAD (
                itad_id_texto TEXT,
                id_tienda INTEGER,
                precio REAL,
                corte_descuento INTEGER,
                fecha_unix INTEGER,
                PRIMARY KEY (itad_id_texto, id_tienda, fecha_unix)
            )
        """)
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS Datos_Actuales_ITAD (
                itad_id_texto TEXT PRIMARY KEY,
                precio_actual REAL,
                tienda_actual INTEGER,
                precio_minimo REAL,
                tienda_minimo INTEGER,
                fecha_minimo INTEGER,
                en_bundle INTEGER DEFAULT 0,
                fecha_actualizacion INTEGER
            )
        """)
        conn.commit()
        print('Tablas listas.')
    finally:
        conn.close()


In [14]:
preparar_tablas_itad()

Tablas listas.


In [15]:
def buscar_ids_itad(limite=50):
    conn = sqlite3.connect(DB_PATH)
    try:
        juegos = conn.execute("""
            SELECT j.juego_id, j.titulo
            FROM CAT_Juego j
            LEFT JOIN REL_Juego_ITAD r ON j.juego_id = r.juego_id
            WHERE j.id_steam IS NOT NULL
              AND r.itad_id_texto IS NULL
              AND j.categoria = 0
            LIMIT ?
        """, (limite,)).fetchall()
    finally:
        conn.close()

    if not juegos:
        print('No hay más juegos por vincular.')
        return

    print(f'{len(juegos)} juegos a vincular')

    for juego_id, titulo in juegos:
        try:
            res = requests.get(
                'https://api.isthereanydeal.com/games/search/v1',
                params={'key': ITAD_KEY, 'title': titulo, 'limit': 1},
                timeout=10
            )

            if res.status_code == 200:
                data = res.json()
                if data:
                    itad_id = data[0].get('id')
                    if itad_id:
                        conn = sqlite3.connect(DB_PATH)
                        try:
                            cursor = conn.cursor()
                            cursor.execute(
                                'INSERT OR IGNORE INTO CAT_ITAD_Juego (itad_id_texto, itad_titulo) VALUES (?, ?)',
                                (itad_id, titulo)
                            )
                            cursor.execute(
                                'INSERT OR IGNORE INTO REL_Juego_ITAD (juego_id, itad_id_texto) VALUES (?, ?)',
                                (juego_id, itad_id)
                            )
                            conn.commit()
                            print(f'  OK: {titulo} -> {itad_id}')
                        finally:
                            conn.close()
                    else:
                        print(f'  sin ID: {titulo}')
                else:
                    print(f'  no encontrado: {titulo}')

            elif res.status_code == 401:
                print('  Error 401 - revisar API key')
                break
            elif res.status_code == 429:
                print('  Rate limit - esperando 20s')
                time.sleep(20)
                continue

        except Exception as e:
            print(f'  Error con {titulo}: {e}')

        time.sleep(1)

In [ ]:
#Prueba version 6 de la funcion
#buscar_ids_itad(limite=10)


In [ ]:
# Loop completo hasta obtenrlos a  todos
while True:
    conn = sqlite3.connect(DB_PATH)
    try:
        pendientes = conn.execute("""
            SELECT COUNT(*)
            FROM CAT_Juego j
            LEFT JOIN REL_Juego_ITAD r ON j.juego_id = r.juego_id
            WHERE j.id_steam IS NOT NULL
              AND r.itad_id_texto IS NULL
              AND j.categoria = 0
        """).fetchone()[0]
    finally:
        conn.close()

    if pendientes == 0:
        print('Todos los juegos vinculados.')
        break

    print(f'Faltan {pendientes} juegos')
    buscar_ids_itad(limite=100)
    print('Pausa')
    time.sleep(30)

Todos los juegos vinculados.


In [ ]:
#historial de precios desde una fecha
def _fetch_historial(itad_id, since_iso):
    res = requests.get(
        f'{ITAD_BASE}/games/history/v2',
        params={'key': ITAD_KEY, 'id': itad_id, 'country': 'MX', 'since': since_iso},
        timeout=TIMEOUT
    )
    return res.json() if res.status_code == 200 else []

#encontrar le minimo
def _fetch_low(itad_id):
    res = requests.post(
        f'{ITAD_BASE}/games/historylow/v1?key={ITAD_KEY}&country=MX',
        json=[itad_id], timeout=TIMEOUT
    )
    if res.status_code == 200 and res.json():
        low = res.json()[0].get('low')
        if low:
            ts = int(datetime.datetime.fromisoformat(low['timestamp']).timestamp()) if low.get('timestamp') else None
            return low.get('price', {}).get('amount'), low.get('shop', {}).get('id'), ts
    return None, None, None

#precio actual en tienda mas barata
def _fetch_precio_actual(itad_id):
    res = requests.post(
        f'{ITAD_BASE}/games/prices/v2?key={ITAD_KEY}&country=MX',
        json=[itad_id], timeout=TIMEOUT
    )
    if res.status_code == 200 and res.json():
        deals = res.json()[0].get('deals')
        if deals:
            return deals[0]['price']['amount'], deals[0]['shop']['id']
    return None, None

#si esta en budle el juego
#PArece que la API no funciona bien para este ya que da vacios.
def _fetch_bundle(itad_id):
    res = requests.post(
        f'{ITAD_BASE}/games/bundles/v2?key={ITAD_KEY}',
        json=[itad_id], timeout=TIMEOUT
    )
    if res.status_code == 200:
        data = res.json()
        if data and isinstance(data[0].get('bundles'), list):
            return 1 if len(data[0]['bundles']) > 0 else 0
    return 0

#usar hilos porque tarda mucho este proceso en las primeras versiones
def _procesar_juego_itad(itad_id, since_iso):
    with ThreadPoolExecutor(max_workers=4) as ex:
        fut_hist  = ex.submit(_fetch_historial, itad_id, since_iso)
        fut_low = ex.submit(_fetch_low, itad_id)
        fut_actual = ex.submit(_fetch_precio_actual, itad_id)
        fut_bundle = ex.submit(_fetch_bundle, itad_id)

        historial = fut_hist.result()
        precio_min, tienda_min, fecha_min  = fut_low.result()
        precio_act, tienda_act  = fut_actual.result()
        en_bundle = fut_bundle.result()

    return historial, precio_min, tienda_min, fecha_min, precio_act, tienda_act, en_bundle

In [ ]:
#descaragr el historial de juegos que no se aun procesado, es mejor limpiar la tabla y volver a correr
# meses_atras: cuántos meses hacia atrás pedir el historial de precios.
def procesar_lote_itad(meses_atras=1, limite_juegos=10):
    
    conn = sqlite3.connect(DB_PATH)
    conn.execute('PRAGMA journal_mode=WAL')
    conn.execute('PRAGMA synchronous=NORMAL')
    cursor = conn.cursor()

    try:
        # Solo toma juegos que aún no tienen entrada en Datos_Actuales_ITAD
        cursor.execute("""
            SELECT r.itad_id_texto
            FROM REL_Juego_ITAD r
            LEFT JOIN Datos_Actuales_ITAD d ON r.itad_id_texto = d.itad_id_texto
            WHERE d.itad_id_texto IS NULL
            LIMIT ?
        """, (limite_juegos,))

        juegos = [row[0] for row in cursor.fetchall()]

        if not juegos:
            print('Sin juegos pendientes.')
            return False

        print(f'{len(juegos)} juegos a procesar')

        fecha_dt  = datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(days=meses_atras * 30)
        since_iso = fecha_dt.strftime('%Y-%m-%dT%H:%M:%SZ')

        for itad_id in juegos:
            print(f'\n[{itad_id}]')
            try:
                historial, precio_min, tienda_min, fecha_min, precio_act, tienda_act, en_bundle = \
                    _procesar_juego_itad(itad_id, since_iso)

                # Insertar tiendas nuevas encontradas en el historial
                tiendas_nuevas = {
                    (c['shop']['id'], c['shop'].get('name', 'Desconocida'))
                    for c in historial
                    if c.get('shop', {}).get('id')
                }
                if tiendas_nuevas:
                    cursor.executemany(
                        'INSERT OR IGNORE INTO CAT_Tienda (id_tienda, nombre) VALUES (?, ?)',
                        tiendas_nuevas
                    )

                hist_rows = []
                for cambio in historial:
                    t_id      = cambio.get('shop', {}).get('id')
                    deal      = cambio.get('deal', {})
                    precio    = deal.get('price', {}).get('amount')
                    cut       = deal.get('cut', 0)
                    fecha_str = cambio.get('timestamp')
                    if t_id and precio and fecha_str:
                        ts = int(datetime.datetime.fromisoformat(fecha_str).timestamp())
                        hist_rows.append((itad_id, t_id, precio, cut, ts))

                if hist_rows:
                    cursor.executemany("""
                        INSERT OR IGNORE INTO Hist_Precios_ITAD
                        (itad_id_texto, id_tienda, precio, corte_descuento, fecha_unix)
                        VALUES (?, ?, ?, ?, ?)
                    """, hist_rows)

                print(f'  historial: {len(hist_rows)} registros')
                print(f'  precio actual: {precio_act} | mínimo: {precio_min} | bundle: {en_bundle}')

                cursor.execute("""
                    INSERT OR REPLACE INTO Datos_Actuales_ITAD
                    (itad_id_texto, precio_actual, tienda_actual, precio_minimo,
                     tienda_minimo, fecha_minimo, en_bundle, fecha_actualizacion)
                    VALUES (?, ?, ?, ?, ?, ?, ?, strftime('%s','now'))
                """, (itad_id, precio_act, tienda_act, precio_min, tienda_min, fecha_min, en_bundle))

                conn.commit()

            except Exception as e:
                print(f'  error en {itad_id}: {e}')
                conn.rollback()

            time.sleep(0.5)

        return True

    finally:
        conn.close()
        print('Conexión cerrada.')

In [21]:
while True:
    continuar = procesar_lote_itad(meses_atras=10, limite_juegos=200)
    if not continuar:
        break
    print('Pausa')
    time.sleep(5)

200 juegos a procesar

[018d937f-3a3b-7210-bd2d-0d1dfb1d84c0]
  historial: 356 registros
  precio actual: 14.99 | mínimo: 12.29 | bundle: 0

[018d937f-4964-7357-a557-f04dac608915]
  historial: 291 registros
  precio actual: 19.99 | mínimo: 6.83 | bundle: 0

[018d937f-012f-73b8-ab2c-898516969e6a]
  historial: 11 registros
  precio actual: None | mínimo: 0.99 | bundle: 0

[018d937e-f877-71b3-aba5-8ff11818cd0b]
  historial: 285 registros
  precio actual: 2.99 | mínimo: 2.98 | bundle: 0

[018d937f-3497-71d8-91b5-442e6b36f0cd]
  historial: 162 registros
  precio actual: 4.99 | mínimo: 2.15 | bundle: 0

[018d937f-59ac-7213-8030-15ae80b3b930]
  historial: 0 registros
  precio actual: None | mínimo: None | bundle: 0

[018d937f-012f-73b8-ab2c-898516969e6a]
  historial: 11 registros
  precio actual: None | mínimo: 0.99 | bundle: 0

[018d937f-4d3f-72ee-b598-97cbfa030c47]
  historial: 39 registros
  precio actual: None | mínimo: 3.74 | bundle: 0

[018d937e-f877-71b3-aba5-8ff11818cd0b]
  historial:

In [ ]:
# ver cuantos nos faltan
conn = sqlite3.connect(DB_PATH)
try:
    # Juegos sin vincular
    df_pendientes = pd.read_sql("""
        SELECT COUNT(*) AS sin_vincular
        FROM CAT_Juego j
        LEFT JOIN REL_Juego_ITAD r ON j.juego_id = r.juego_id
        WHERE j.id_steam IS NOT NULL
          AND r.itad_id_texto IS NULL
          AND j.categoria = 0
    """, conn)

    # Estado de tablas ITAD
    df_estado = pd.read_sql("""
        SELECT
            (SELECT COUNT(*) FROM CAT_ITAD_Juego)     AS ids_itad,
            (SELECT COUNT(*) FROM Datos_Actuales_ITAD) AS juegos_procesados,
            (SELECT COUNT(*) FROM Hist_Precios_ITAD)   AS registros_hist,
            (SELECT COUNT(*) FROM CAT_Tienda)           AS tiendas
    """, conn)

    display(df_pendientes)
    display(df_estado)
finally:
    conn.close()

,COUNT(*)
0,0


In [57]:
with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql("""
        SELECT
            COUNT(j.itad_id_texto)                  AS juegos_vinculados,
            COUNT(d.itad_id_texto)                  AS juegos_procesados,
            COUNT(h.itad_id_texto)                  AS juegos_con_historial,
            SUM(CASE WHEN d.precio_actual IS NOT NULL THEN 1 ELSE 0 END) AS con_precio_actual,
            SUM(CASE WHEN d.precio_minimo IS NOT NULL THEN 1 ELSE 0 END) AS con_precio_minimo,
            (SELECT COUNT(*) FROM Hist_Precios_ITAD) AS total_registros_hist
        FROM CAT_Juego j
        LEFT JOIN Datos_Actuales_ITAD d ON j.itad_id_texto = d.itad_id_texto
        LEFT JOIN (SELECT DISTINCT itad_id_texto FROM Hist_Precios_ITAD) h ON j.itad_id_texto = h.itad_id_texto
        WHERE j.itad_id_texto IS NOT NULL
    """, conn)
    display(df)

,juegos_vinculados,juegos_procesados,juegos_con_historial,con_precio_actual,con_precio_minimo,total_registros_hist
0,7172,195,158,156,155,2457


In [ ]:
conexion = sqlite3.connect(DB_PATH)
tablas = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';", conexion)
resumen = []
for tabla in tablas['name']:
    cantidad = pd.read_sql_query(f"SELECT COUNT(*) as Cantidad FROM {tabla};", conexion).iloc[0,0]
    resumen.append({'Nombre': tabla, 'Cantidad': cantidad})
df = pd.DataFrame(resumen).sort_values(by='Cantidad', ascending=False)
display(df)

conexion.close()

,Nombre,Cantidad
20,Hist_Steam_Reviews,600194
17,REL_Juego_Etiqueta,127396
19,REL_Juegos_Similares,101550
18,REL_Juego_Plataforma,38040
10,REL_Juego_Genero,27717
11,REL_Juego_Tematica,21669
12,REL_Juego_Modo,17180
15,REL_Juego_Editor,14017
14,REL_Juego_Desarrollador,11338
13,REL_Juego_Perspectiva,11155
